[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tayalmanan28/MuJoCo-Tutorial/blob/main/tutorial/07_mjx.ipynb)

⚡ **This notebook requires a GPU runtime.** In Colab: Runtime → Change runtime type → T4 GPU

# Tutorial 7: MJX — GPU-Accelerated Simulation

MJX is MuJoCo's JAX backend for massively parallel simulation on GPU.
Perfect for reinforcement learning where you need thousands of environments.

In [1]:
!pip install -q mujoco mujoco-mjx jax[cuda12] mediapy

In [2]:
import os
os.environ["MUJOCO_GL"] = "egl"  # 强制使用离屏渲染，绕过 GUI 依赖
import mujoco
from mujoco import mjx
import jax
import jax.numpy as jnp
import numpy as np
import mediapy as media
import time

print(f"JAX devices: {jax.devices()}")
print(f"MuJoCo version: {mujoco.__version__}")

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'


ERROR:jax._src.xla_bridge:Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/jax/_src/xla_bridge.py", line 495, in discover_pjrt_plugins
    plugin_module.initialize()
    ~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/jax_plugins/xla_cuda12/__init__.py", line 373, in initialize
    c_api = xb.register_plugin(
        'cuda', priority=500, library_path=str(path), options=options
    )
  File "/usr/local/lib/python3.13/dist-packages/jax/_src/xla_bridge.py", line 637, in register_plugin
    c_api = xla_client.load_pjrt_plugin_dynamically(plugin_name, library_path)
  File "/usr/local/lib/python3.13/dist-packages/jaxlib/xla_client.py", line 115, in load_pjrt_plugin_dynamically
    return _xla.load_pjrt_plugin(plugin_name, library_path, c_api=None)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
jax.errors.JaxRuntimeEr

JAX devices: [CudaDevice(id=0)]
MuJoCo version: 3.15.0


In [3]:
x = jnp.ones((1000, 1000))
y = jnp.dot(x, x)
print(y.device)

cuda:0


## MJX Basics

MJX converts MuJoCo models to JAX-compatible pytrees that run on GPU:

1. Create a standard `MjModel`
2. Convert to MJX with `mjx.put_model()`
3. Use `mjx.step()` instead of `mujoco.mj_step()`

In [12]:
xml = """
<mujoco>
  <option gravity="0 0 -9.81" timestep="0.002"/>
  <worldbody>
    <light directional="true" diffuse=".8 .8 .8" specular=".2 .2 .2" pos="0 0 5" dir="0 0 -1"/>
    <geom type="plane" size="5 5 0.1"/>
    <body pos="0 0 1">
      <joint type="free"/>
      <geom type="sphere" size="0.1" mass="1"/>
    </body>
  </worldbody>
</mujoco>
"""

# Standard MuJoCo model
mj_model = mujoco.MjModel.from_xml_string(xml)
mj_data = mujoco.MjData(mj_model)

# Convert to MJX
mjx_model = mjx.put_model(mj_model)
mjx_data = mjx.put_data(mj_model, mj_data)

print(f"MJX model type: {type(mjx_model)}")
print(f"MJX data qpos: {mjx_data.qpos}")

MJX model type: <class 'mujoco.mjx._src.types.Model'>
MJX data qpos: [0. 0. 1. 1. 0. 0. 0.]


## JIT-Compiled Stepping

In [5]:
# JIT compile the step function
jit_step = jax.jit(mjx.step)

# First call triggers compilation
mjx_data = jit_step(mjx_model, mjx_data)

# Benchmark: 1000 steps
t0 = time.time()
for _ in range(1000):
    mjx_data = jit_step(mjx_model, mjx_data)
mjx_data.qpos.block_until_ready()  # wait for GPU
elapsed = time.time() - t0

print(f"1000 steps in {elapsed*1000:.1f}ms ({1000/elapsed:.0f} steps/sec)")
print(f"Final height: {float(mjx_data.qpos[2]):.4f}")

1000 steps in 2552.8ms (392 steps/sec)
Final height: 0.0996


## Batched (Parallel) Simulation

The real power of MJX: simulate thousands of environments simultaneously using `jax.vmap`.

In [6]:
batch_size = 4096

# Create batch of initial states with random velocities
rng = jax.random.PRNGKey(0)
random_vels = jax.random.normal(rng, (batch_size, 6)) * jnp.array([2, 2, 5, 0.5, 0.5, 0.5])

# Reset base data
mj_data_reset = mujoco.MjData(mj_model)
mjx_data_base = mjx.put_data(mj_model, mj_data_reset)

# Batch: each environment gets a different initial velocity
batched_data = jax.vmap(lambda v: mjx_data_base.replace(qvel=v))(random_vels)

print(f"Batched qpos shape: {batched_data.qpos.shape}")  # (4096, 7)
print(f"Simulating {batch_size} environments in parallel...")

Batched qpos shape: (4096, 7)
Simulating 4096 environments in parallel...


In [7]:
# Batched step function
@jax.jit
def batched_step(data):
    return jax.vmap(mjx.step, in_axes=(None, 0))(mjx_model, data)

# Simulate 500 steps for all environments
n_steps = 1000

t0 = time.time()
data = batched_data
for _ in range(n_steps):
    data = batched_step(data)
data.qpos.block_until_ready()
elapsed = time.time() - t0

total_steps = batch_size * n_steps
print(f"{total_steps:,} total steps in {elapsed:.2f}s")
print(f"Throughput: {total_steps/elapsed:,.0f} steps/sec")
print(f"\nFinal heights — mean: {float(data.qpos[:, 2].mean()):.3f}, std: {float(data.qpos[:, 2].std()):.3f}")

4,096,000 total steps in 26.07s
Throughput: 157,134 steps/sec

Final heights — mean: 0.222, std: 0.908


## Rendering MJX Results

MJX runs on GPU but rendering uses CPU MuJoCo. Transfer data back:

In [13]:

# Simulate one trajectory and render
mj_data_viz = mujoco.MjData(mj_model)
mjx_data_viz = mjx.put_data(mj_model, mj_data_viz)
mjx_data_viz = mjx_data_viz.replace(qvel=jnp.array([3.0, 0, 8.0, 0, 0, 0]))

renderer = mujoco.Renderer(mj_model, height=480, width=600)
frames = []

for i in range(300):
    mjx_data_viz = jit_step(mjx_model, mjx_data_viz)

    if i % 5 == 0:  # render every 5th step (for 60fps at dt=0.002)
        # Transfer back to CPU for rendering
        mjx.get_data_into(mj_data_viz, mj_model, mjx_data_viz)
        renderer.update_scene(mj_data_viz)
        frames.append(renderer.render().copy())

renderer.close()
media.show_video(frames, fps=30)

## When to Use MJX vs CPU MuJoCo

| Use Case | Recommendation |
|----------|---------------|
| Single environment, interactive | CPU MuJoCo + `mujoco.viewer` |
| RL training (PPO, SAC) | MJX — batch 1000s of envs |
| System identification | MJX — differentiable physics |
| Trajectory optimization | MJX — autodiff through dynamics |
| Complex contact-rich tasks | CPU MuJoCo (MJX has some contact limitations) |

### Key Differences
- MJX supports most but not all MuJoCo features (check [docs](https://mujoco.readthedocs.io/en/stable/mjx.html#feature-parity))
- MJX is **differentiable** — you can compute gradients through the physics
- Ideal batch size: 1024–8192 for best GPU utilization